# Cook Inlet SFINCS - building a model in Alaska

Builds and runs a SFINCS model for Cook Inlet, Alaska. The build inputs - an
AOI polygon, the flowpaths entering it, and the quadtree refinement zones -
were all drawn in QGIS with the `nwm_coastal` plugin, and sections 1 to 3
show how.

**These are suggested steps.** Any method of producing a polygon GeoJSON is
fine; nothing below depends on having used the plugin.

## Setup

In [1]:
from __future__ import annotations

import os
import subprocess
import sys
import urllib.request
from pathlib import Path

notebook_dir = Path.cwd()  # assumes run from docs/examples/notebooks/
example_dir = (notebook_dir.parent / "alaska_sfincs").resolve()
os.chdir(example_dir)


def cli(*args: str) -> None:
    """Run a coastal-calibration CLI command and stream its output."""
    print(f"$ coastal-calibration {' '.join(args)}\n")
    subprocess.run([sys.executable, "-m", "coastal_calibration.cli", *args], check=True)


print(f"Working directory: {example_dir}")
print("Inputs:", sorted(p.name for p in example_dir.glob("*.geojson")))

Working directory: /home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_sfincs
Inputs: ['sfincs_alaska_cookinlet.geojson', 'sfincs_alaska_flowpaths_ngen.geojson', 'sfincs_alaska_flowpaths_nwm.geojson', 'sfincs_refine_1.geojson', 'sfincs_refine_2.geojson', 'sfincs_refine_3.geojson', 'sfincs_refine_4.geojson', 'sfincs_refine_5.geojson', 'sfincs_refine_6.geojson', 'sfincs_refine_7.geojson']


## 1. The AOI polygon

Sketch a polygon around the area of interest, using the NHF catchment divides
as a guide.

Union it with the divides and look at the result.

<div style="display: flex; gap: 1em; align-items: flex-start; flex-wrap: wrap;">
  <figure style="flex: 1.89; margin: 0;">
    <img src="../images/Alaska_SFINCS_initial_sketcher_poly.jpg" alt="Initial sketched polygon over Cook Inlet" style="width: 100%;">
    <figcaption>Initial sketched polygon over Cook Inlet</figcaption>
  </figure>
  <figure style="flex: 1.84; margin: 0;">
    <img src="../images/Alaska_SFINCS_intial_sketcher_merge.jpg" alt="Polygon merged with divides" style="width: 100%;">
    <figcaption>Polygon merged with divides</figcaption>
  </figure>
</div>

Edit the polygon to drop divides you do not want, re-merging as you go.

<div style="display: flex; gap: 1em; align-items: flex-start; flex-wrap: wrap;">
  <figure style="flex: 1.76; margin: 0;">
    <img src="../images/Alaska_SFINCS_edit_sketcher.jpg" alt="Editing the sketched polygon" style="width: 100%;">
    <figcaption>Editing the sketched polygon</figcaption>
  </figure>
  <figure style="flex: 1.74; margin: 0;">
    <img src="../images/Alaska_SFINCS_edit_merged.jpg" alt="Re-merged after editing" style="width: 100%;">
    <figcaption>Re-merged after editing</figcaption>
  </figure>
</div>

Some nearshore areas have no divides at all; trace those by hand so the
polygon follows the coast.

<div style="display: flex; gap: 1em; align-items: flex-start; flex-wrap: wrap;">
  <figure style="flex: 1.81; margin: 0;">
    <img src="../images/Alaksa_SFINCS_missing_divide.jpg" alt="Nearshore area with no divide" style="width: 100%;">
    <figcaption>Nearshore area with no divide</figcaption>
  </figure>
  <figure style="flex: 1.57; margin: 0;">
    <img src="../images/Alaska_SFINCS_include_missing_divide.jpg" alt="Polygon extended over the missing divide" style="width: 100%;">
    <figcaption>Polygon extended over the missing divide</figcaption>
  </figure>
</div>

Merging can leave invalid geometry - self-intersections and the like. Check
validity and fix it with QGIS's own geometry tools before saving. The result
is `sfincs_alaska_cookinlet.geojson`.

![Final Cook Inlet AOI](../images/Alaska_SFINCS_final.jpg)

## 2. Flowpaths entering the domain

Load the NWM v3 geodatabase or an NHF GeoPackage through the plugin, select
the flowpaths that cross into the AOI, and export them.

In Alaska the NWM carries far more flowpaths than the NHF hydrofabric does,
which makes the NWM set hard to choose from on its own. Here the NHF set was
used to guide which NWM reaches to keep.

<div style="display: flex; gap: 1em; align-items: flex-start; flex-wrap: wrap;">
  <figure style="flex: 1.75; margin: 0;">
    <img src="../images/SFINCS_NWM_vs_NGEN_flowpaths.jpg" alt="NWM and NGEN flowpaths compared" style="width: 100%;">
    <figcaption>NWM and NGEN flowpaths compared</figcaption>
  </figure>
  <figure style="flex: 0.86; margin: 0;">
    <img src="../images/SFINCS_example_selecting_flowpaths.jpg" alt="Selecting the flowpaths entering the domain" style="width: 100%;">
    <figcaption>Selecting the flowpaths entering the domain</figcaption>
  </figure>
</div>

Both sets are shipped. `create.yaml` uses the NWM one, keyed on its `ID`
column; switch `river_discharge.flowlines` to the `_ngen` file and
`nwm_id_column` to `fp_id` for a NextGen-keyed run.

In [2]:
import geopandas as gpd

for name in ("sfincs_alaska_flowpaths_nwm.geojson", "sfincs_alaska_flowpaths_ngen.geojson"):
    g = gpd.read_file(name)
    print(f"{name}: {len(g)} flowpaths, {g.crs}")

sfincs_alaska_flowpaths_nwm.geojson: 97 flowpaths, EPSG:4269
sfincs_alaska_flowpaths_ngen.geojson: 110 flowpaths, EPSG:3338


## 3. Refinement zones

It's common to use additional GIS data layers to help determine areas
where higher resolution may be warranted. In this case, a Alaska Geoportal
datasets of USA structures and roads in Alaska were used to help
define some refinement areas (https://alaska-geoportal-soa-dnr.hub.arcgis.com/)

![Using additional data to place refinement zones](../images/SFINCS_use_adtnl_data_refine.jpg)

QGIS's `buffer` around the flowpaths is another way to raise resolution along
the rivers. Four steps, starting from the flowpaths layer: clip them to the
AOI, buffer the result, split the buffer into single parts, and save it as a
refinement polygon.

<div style="display: flex; gap: 1em; align-items: flex-start; flex-wrap: wrap;">
  <figure style="flex: 2.01; margin: 0;">
    <img src="../images/SFINCS_make_buffered_rivers_clip_step.jpg" alt="1. Clip the flowpaths to the AOI" style="width: 100%;">
    <figcaption>1. Clip the flowpaths to the AOI</figcaption>
  </figure>
  <figure style="flex: 1.98; margin: 0;">
    <img src="../images/SFINCS_make_buffered_rivers_buffer_step.jpg" alt="2. Buffer the clipped flowpaths" style="width: 100%;">
    <figcaption>2. Buffer the clipped flowpaths</figcaption>
  </figure>
</div>

<div style="display: flex; gap: 1em; align-items: flex-start; flex-wrap: wrap;">
  <figure style="flex: 1.99; margin: 0;">
    <img src="../images/SFINCS_make_buffered_rivers_multi_to_single_step.jpg" alt="3. Multipart to single part" style="width: 100%;">
    <figcaption>3. Multipart to single part</figcaption>
  </figure>
  <figure style="flex: 0.89; margin: 0;">
    <img src="../images/SFINCS_make_buffered_rivers_save_step.jpg" alt="4. Save as a refinement polygon" style="width: 100%;">
    <figcaption>4. Save as a refinement polygon</figcaption>
  </figure>
</div>

The seven `sfincs_refine_*.geojson` polygons each refine two levels, taking
1024 m cells down to 256 m.

In [3]:
for p in sorted(example_dir.glob("sfincs_refine_*.geojson")):
    g = gpd.read_file(p)
    print(f"{p.name}: {g.to_crs(3338).area.sum() / 1e6:.0f} km2")

sfincs_refine_1.geojson: 51 km2
sfincs_refine_2.geojson: 2268 km2
sfincs_refine_3.geojson: 2396 km2
sfincs_refine_4.geojson: 347 km2
sfincs_refine_5.geojson: 85 km2
sfincs_refine_6.geojson: 424 km2
sfincs_refine_7.geojson: 149 km2


## 4. Elevation

NCEI's regional DEMs cover Cook Inlet, over plain HTTPS.

Two are used here, finest first:

| file | resolution | extent |
| --- | --- | --- |
| `cook_inlet_ak_3_mhhw_2020.nc` | 3" (~90 m) | 60.20-61.70 N, upper inlet only |
| `cook_inlet_ak_8_mhhw_2020.nc` | 8" (~240 m) | 58.40-61.75 N, spans the AOI |

`gebco_15arcs` fills what is left past their footprints.

In [4]:
NCEI = "https://www.ngdc.noaa.gov/thredds/fileServer/regional"
dem_dir = Path("./downloads/dem")
dem_dir.mkdir(parents=True, exist_ok=True)

for name in ("cook_inlet_ak_3_mhhw_2020.nc", "cook_inlet_ak_8_mhhw_2020.nc"):
    dest = dem_dir / name
    if dest.exists():
        print(f"already present: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"downloading {NCEI}/{name}")
        urllib.request.urlretrieve(f"{NCEI}/{name}", dest)
        print(f"  -> {dest.stat().st_size / 1e6:.1f} MB")

already present: downloads/dem/cook_inlet_ak_3_mhhw_2020.nc (22.9 MB)
already present: downloads/dem/cook_inlet_ak_8_mhhw_2020.nc (11.1 MB)


### The vertical datum

Both DEMs are on **MHHW**, not MSL.

`create.yaml` adds `offset: 3.87` to both DEMs, which puts them on MSL at Anchorage so
they merge cleanly with GEBCO and match the STOFS forcing datum. That
number is measured at Anchorage and the separation varies by more than a
metre across the inlet, so it is exact near the city and approximate at the
mouth. For a domain this wide the better approach is to pre-transform the
rasters; the offset is the quick version.

In [5]:
import xarray as xr

for name in ("cook_inlet_ak_3_mhhw_2020.nc", "cook_inlet_ak_8_mhhw_2020.nc"):
    ds = xr.open_dataset(dem_dir / name)
    z = ds["z"]
    print(
        f"{name}: {dict(ds.sizes)} | lat {float(ds.lat.min()):.2f}..{float(ds.lat.max()):.2f}"
        f" | lon {float(ds.lon.min()):.2f}..{float(ds.lon.max()):.2f}"
        f" | z {float(z.min()):.1f}..{float(z.max()):.1f} m MHHW"
    )
    ds.close()

cook_inlet_ak_3_mhhw_2020.nc: {'lat': 1800, 'lon': 4560, 'grid_mapping': 12} | lat 60.20..61.70 | lon -152.60..-148.80 | z -161.5..3445.0 m MHHW
cook_inlet_ak_8_mhhw_2020.nc: {'lat': 1507, 'lon': 2588, 'grid_mapping': 12} | lat 58.40..61.75 | lon -154.50..-148.75 | z -322.6..3418.4 m MHHW


### Land cover

Roughness comes from ESA WorldCover, which ships at 10 m. Over this AOI that
is a 73,500 x 36,830 raster - 2.7 billion pixels - and building the subgrid
tables from it needs about 30 GB of memory.

Resampling to 100 m cuts the raster 100x,
to 27 million pixels, and the class distribution is unchanged. `mode` is the
resampling here because the classes are categorical.

In [6]:
import subprocess as _sp

from coastal_calibration.data.esa_worldcover import fetch_esa_worldcover

lulc_dir = Path("./downloads/lulc")
lulc_tif = lulc_dir / "esa_worldcover.tif"

if lulc_tif.exists():
    print(f"already present: {lulc_tif} ({lulc_tif.stat().st_size / 1e6:.1f} MB)")
else:
    lulc_dir.mkdir(parents=True, exist_ok=True)
    native, _, _ = fetch_esa_worldcover(
        aoi=Path("./sfincs_alaska_cookinlet.geojson"),
        output_dir=lulc_dir,
        catalog_name="esa_worldcover_10m",
    )
    _sp.run(
        [
            "gdalwarp",
            "-tr",
            "0.000833",
            "0.000833",
            "-r",
            "mode",
            "-ot",
            "Byte",
            "-dstnodata",
            "0",
            "-co",
            "COMPRESS=DEFLATE",
            str(native),
            str(lulc_tif),
        ],
        check=True,
        capture_output=True,
    )
    native.unlink()
    print(f"resampled to 100 m: {lulc_tif} ({lulc_tif.stat().st_size / 1e6:.1f} MB)")

already present: downloads/lulc/esa_worldcover.tif (0.9 MB)


In [7]:
import rasterio

with rasterio.open(lulc_tif) as r:
    print(f"{r.width} x {r.height} = {r.width * r.height / 1e6:.1f} Mpx, dtype={r.dtypes[0]}")

7353 x 3684 = 27.1 Mpx, dtype=uint8


## 5. Create the model

1024 m base cells, refined to 256 m inside the seven zones, with 4 subgrid
pixels per cell.

In [8]:
print(Path("create.yaml").read_text())

# Cook Inlet, Alaska - SFINCS model creation from AOI
#
# Elevation is not auto-fetched: no built-in source covers Alaska. The NCEI
# Cook Inlet DEMs are downloaded by ../notebooks/alaska_sfincs.ipynb and
# registered through ./dem_catalog.yml.
#
# Usage:
#   coastal-calibration create create.yaml
#   coastal-calibration create create.yaml --dry-run

aoi: ./sfincs_alaska_cookinlet.geojson
output_dir: ./sfincs_cookinlet
download_dir: ./downloads/grid   # inside the example, so a copied folder is self-contained

grid:
  resolution: 1024       # base resolution in meters
  crs: utm               # auto-detect UTM zone (-> EPSG:32605 here)
  rotated: false
  refinement:
  - polygon: ./sfincs_refine_1.geojson
    level: 2             # 2 refinement steps: 1024->512->256 m
  - polygon: ./sfincs_refine_2.geojson
    level: 2
  - polygon: ./sfincs_refine_3.geojson
    level: 2
  - polygon: ./sfincs_refine_4.geojson
    level: 2
  - polygon: ./sfincs_refine_5.geojson
    level: 2
  - polygon: .

In [9]:
cli("create", "create.yaml")

$ coastal-calibration create create.yaml



Coastal Calibration Workflow                                                    
Start Time: 2026-09-28 05:54:38                                                 
----------------------------------------                                        
Stage: create_grid                                                              
Start Time: 2026-09-28 05:54:38                                                 
  Create SFINCS grid from AOI                                                   
  AOI:                                                                          
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_sfincs/sfincs_alaska_cook
inlet.geojson                                                                   
  Resolution: 1024 m, CRS: utm                                                  


  Refinement: 7 polygon(s), max level 2                                         


  Grid created successfully                                                     
  [✓] COMPLETED (5s)                                                            
----------------------------------------                                        
Stage: create_fetch_data                                                        
Start Time: 2026-09-28 05:54:44                                                 
  Fetch elevation and land cover data for AOI                                   
  Reusing existing gebco_15arcs.tif                                             
  Fetched 1 dataset(s): ['gebco_15arcs']                                        
  [✓] COMPLETED (0s)                                                            
----------------------------------------                                        
Stage: create_elevation                                                         
Start Time: 2026-09-28 05:54:44                                                 
  Add elevation and bathymet

  Elevation created successfully                                                
  [✓] COMPLETED (9s)                                                            
----------------------------------------                                        
Stage: create_mask                                                              
Start Time: 2026-09-28 05:54:54                                                 
  Create active cell mask                                                       
  zmin=None                                                                     


...

  Wrote 22 boundary point(s) to sfincs.bnd (bnd_dist=5000.0)                    
  Boundary cells created successfully                                           
  [✓] COMPLETED (0s)                                                            
----------------------------------------                                        
Stage: create_discharge                                                         
Start Time: 2026-09-28 05:54:54                                                 
  Add river discharge source points                                             
  Read 97 flowpath(s) from sfincs_alaska_flowpaths_nwm.geojson                  
    75004300001228: snapped to active cell (729 m away)                         
    75004300002142: snapped to active cell (497 m away)                         
    75004300003185: snapped to active cell (166 m away)                         
    75004300007360: snapped to active cell (605 m away)                         
    75004300008149: snapped 

    75004300005039: snapped to active cell (500 m away)                         
    75004300006837: snapped to active cell (858 m away)                         
    75004300002027: snapped to active cell (788 m away)                         
    75004300003723: snapped to active cell (159 m away)                         
    75004300007828: snapped to active cell (398 m away)                         
    75004300005478: snapped to active cell (719 m away)                         
    75004300004156: snapped to active cell (500 m away)                         
    75004300004983: snapped to active cell (166 m away)                         
    75004300001833: snapped to active cell (1035 m away)                        
    75004300005844: snapped to active cell (398 m away)                         
    75004300002023: snapped to active cell (745 m away)                         
    75004300002105: snapped to active cell (498 m away)                         
    75004400007997: snapped 

  Subgrid tables created successfully                                           
  [✓] COMPLETED (1m 2s)                                                         
----------------------------------------                                        
Stage: create_obs                                                               
Start Time: 2026-09-28 05:55:57                                                 
  Add observation points                                                        
  Loading cached station metadata from cache/coops_stations_metadata.json       
  Fetching datum information for 4 station(s)                                   
  Fetching data from 4 station(s)                                               


    noaa_9455090: DROPPED, nearest active cell is 65190 m away (exceeds         
max_snap_distance_m=2000)                                                       
    noaa_9455500: snapped to active cell (60 m away)                            
    noaa_9455760: snapped to active cell (109 m away)                           
    noaa_9455920: snapped to active cell (176 m away)                           
  Added 3 NOAA CO-OPS observation point(s)                                      
    noaa_9455500: placed at face center z=-4.928 m (0 m from original)          
    noaa_9455760: placed at face center z=-19.541 m (0 m from original)         
    noaa_9455920: placed at face center z=-14.010 m (256 m from original)       
  Snapped 3 observation point(s) to nearest wet cell                            
  [✓] COMPLETED (0s)                                                            
----------------------------------------                                        
Stage: create_write         

  Model written successfully                                                    
  [✓] COMPLETED (0s)                                                            
----------------------------------------                                        
Workflow COMPLETED | Total Duration: 0:01:19                                    
                                                                                
Timing Summary:                                                                 
----------------------------------------                                        
  [✓] create_grid: 5s                                                           
  [✓] create_fetch_data: 0s                                                     
  [✓] create_elevation: 9s                                                      
  [✓] create_mask: 0s                                                           
  [✓] create_boundary: 0s                                                       
  [✓] create_discharge: 0s  

### What was built

In [10]:
model_dir = Path("sfincs_cookinlet")
print(sorted(p.name for p in model_dir.iterdir()))

['.create_status.json', 'create_progress.json', 'create_result.json', 'gis', 'obs_station_map.json', 'sfincs-create-20260928-055438.log', 'sfincs.bnd', 'sfincs.bzs', 'sfincs.inp', 'sfincs.nc', 'sfincs.obs', 'sfincs_nwm.src', 'sfincs_subgrid.nc', 'subgrid']


## 6. Run it

12 hours, the same window as the Alaska SCHISM example, forced by STOFS.
This is the slow part - everything above is the model build.

In [11]:
cli("run", "run.yaml", "--dry-run")

$ coastal-calibration run run.yaml --dry-run



Dry run mode - validating configuration...                                      


Dry run mode - validation passed, no execution                                  
Workflow completed successfully.                                                


In [12]:
cli("run", "run.yaml")

$ coastal-calibration run run.yaml



Coastal Calibration Workflow                                                    
Start Time: 2026-09-28 05:56:09                                                 
----------------------------------------                                        
Cleaned generated files from                                                    
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_sfincs/run               
Stage: download                                                                 
Start Time: 2026-09-28 05:56:09                                                 
  Download input data (NWM, STOFS)                                              


STOFS: using cycle 2026-08-15 00:00:00 for start 2026-08-15 00:00:00            


  meteo/nwm_ana: 13/13 [OK]                                                     
  hydro/nwm: 13/13 [OK]                                                         
  coastal/stofs: 1/1 [OK]                                                       
  Total: 27/27 (failed: 0)                                                      
  Download complete. Raw files stored in                                        
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_sfincs/downloads/forcing 
  [✓] COMPLETED (5m 3s)                                                         
----------------------------------------                                        
Stage: sfincs_symlinks                                                          
Start Time: 2026-09-28 06:01:12                                                 
  Create .nc symlinks for NWM data                                              
  Created 13 meteo + 13 streamflow symlinks in                                  
/home/laurscham/ngwpc/nwm-co

  Copied pre-built model from                                                   
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_sfincs/sfincs_cookinlet  
to                                                                              
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_sfincs/run/sfincs_model  


...

  Loading cached station metadata from cache/coops_stations_metadata.json       
  Requesting water_level data for 3 station(s) from 20260815 00:00 to 20260815  
12:00                                                                           
  Fetching data from 3 station(s)                                               


  Successfully retrieved data for 3 station(s)                                  


  Loading cached station metadata from cache/coops_stations_metadata.json       
  Fetching datum information for 3 station(s)                                   
  Fetching data from 3 station(s)                                               


  Saved 1 comparison figure(s) to                                               
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_sfincs/run/sfincs_model/f
igs                                                                             
  Including 3 NOAA gauge(s) from obs_station_map.json                           


  Wrote obs_water_level.parquet with 3 station(s) and 13 timestep(s)            
  [✓] COMPLETED (4s)                                                            
----------------------------------------                                        
Workflow COMPLETED | Total Duration: 0:05:58                                    
                                                                                
Timing Summary:                                                                 
----------------------------------------                                        
  [✓] download: 5m 3s                                                           
  [✓] sfincs_symlinks: 0s                                                       
  [✓] sfincs_data_catalog: 0s                                                   
  [✓] sfincs_init: 1s                                                           
  [✓] sfincs_timing: 0s                                                         
  [✓] sfincs_forcing: 12s   

## Summary

The build inputs were drawn in QGIS, the elevation came from NCEI over plain
HTTPS through a data catalog, and `create` plus `run` did the rest.